In [1]:
from minio import Minio
from dotenv import load_dotenv
from pyspark import SparkContext, SparkConf, SQLContext
import os
import requests

load_dotenv()

False

In [2]:
client_minio = Minio(
    f"{os.getenv("minio_ip_address")}:3900",
    access_key=os.getenv("key_id"),
    secret_key=os.getenv("secret_key"),
    secure=False,
    region="garage",
)

conf = SparkConf() \
    .setAppName('SparkApp') \
    .setMaster('spark://spark:7077') \
    .set("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.spark:spark-hadoop-cloud_2.12:3.5.3,graphframes:graphframes:0.8.3-spark3.5-s_2.12") \
    .set("spark.hadoop.fs.s3a.committer.name", "staging") \
    .set("spark.hadoop.mapreduce.outputcommitter.factory.scheme.s3a", "org.apache.hadoop.fs.s3a.commit.S3ACommitterFactory") \
    .set("spark.hadoop.fs.s3a.committer.staging.tmp.path", "/tmp/s3a-commit") \
    .set("spark.hadoop.fs.s3a.committer.staging.unique-filenames", "true")\
    .set("spark.hadoop.fs.s3a.committer.staging.conflict-mode", "replace") # utilisé pour le stockage 
sc = SparkContext(conf=conf)

sc._jsc.hadoopConfiguration().set("fs.s3a.endpoint", f"http://{os.getenv('minio_ip_address')}:3900")
sc._jsc.hadoopConfiguration().set("fs.s3a.access.key", os.getenv('key_id')) # set key ID 
sc._jsc.hadoopConfiguration().set("fs.s3a.endpoint.region", "garage")
sc._jsc.hadoopConfiguration().set("fs.s3a.secret.key", os.getenv('secret_key')) # set secret key
sc._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")
sc._jsc.hadoopConfiguration().set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
sc._jsc.hadoopConfiguration().set("fs.s3a.connection.ssl.enabled", "false")

sql_context = SQLContext(sc)

:: loading settings :: url = jar:file:/opt/conda/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
org.apache.spark#spark-hadoop-cloud_2.12 added as a dependency
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-058bee55-493e-4793-a26a-1a9ca409632e;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found org.apache.spark#spark-hadoop-cloud_2.12;3.5.3 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.cod

In [3]:
client_minio.fput_object(os.getenv("bucket_name"), "IDFM-gtfs/routes.txt", "/data/IDFM-gtfs/routes.txt")
client_minio.fput_object(os.getenv("bucket_name"), "IDFM-gtfs/stops.txt", "/data/IDFM-gtfs/stops.txt")
client_minio.fput_object(os.getenv("bucket_name"), "IDFM-gtfs/stop_times.txt", "/data/IDFM-gtfs/stop_times.txt")
client_minio.fput_object(os.getenv("bucket_name"), "IDFM-gtfs/trips.txt", "/data/IDFM-gtfs/trips.txt")

In [4]:
routes_df = sql_context.read.csv("s3a://graphx/IDFM-gtfs/routes.txt", header=True, inferSchema=True)
routes_df.createOrReplaceTempView("routes")
stops_df = sql_context.read.csv("s3a://graphx/IDFM-gtfs/stops.txt", header=True, inferSchema=True)
stops_df.createOrReplaceTempView("stops")
stops_times_df = sql_context.read.csv("s3a://graphx/IDFM-gtfs/stop_times.txt", header=True, inferSchema=True)
stops_times_df.createOrReplaceTempView("stop_times")
trips_df = sql_context.read.csv("s3a://graphx/IDFM-gtfs/trips.txt", header=True, inferSchema=True)
trips_df.createOrReplaceTempView("trips")

26/03/22 14:28:50 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


#### Get all subway routes

In [5]:
subway_routes_df = sql_context.sql(
"""
    SELECT DISTINCT * FROM routes r
    WHERE agency_id = "IDFM:Operator_100"  
        AND route_type = 1
""")

subway_routes_df.show()

subway_routes_df.createOrReplaceTempView("subway_routes")

+-----------+-----------------+----------------+---------------+----------+----------+---------+-----------+----------------+----------------+
|   route_id|        agency_id|route_short_name|route_long_name|route_desc|route_type|route_url|route_color|route_text_color|route_sort_order|
+-----------+-----------------+----------------+---------------+----------+----------+---------+-----------+----------------+----------------+
|IDFM:C01379|IDFM:Operator_100|               9|              9|      NULL|         1|     NULL|     B6BD00|          000000|            NULL|
|IDFM:C01376|IDFM:Operator_100|               6|              6|      NULL|         1|     NULL|     6ECA97|          000000|            NULL|
|IDFM:C01381|IDFM:Operator_100|              11|             11|      NULL|         1|     NULL|     704B1C|          FFFFFF|            NULL|
|IDFM:C01371|IDFM:Operator_100|               1|              1|      NULL|         1|     NULL|     FFBE00|          000000|            NULL|

#### Get all trips from subway routes

In [6]:
subway_trips_df = sql_context.sql(
    """SELECT DISTINCT
        t.trip_id,
        r.route_id,
        r.route_long_name,
        st.stop_sequence,
        s.stop_name,
        s.stop_id,
        s.stop_lon,
        s.stop_lat,
        parent_station,
        t.direction_id
    FROM trips t
    JOIN subway_routes r ON t.route_id = r.route_id
    JOIN stop_times st ON t.trip_id = st.trip_id
    JOIN stops s ON st.stop_id = s.stop_id
    ORDER BY t.trip_id, st.stop_sequence"""
)

subway_trips_df.createOrReplaceTempView("subway_trips")

In [7]:
subway_trips_df.show(10)

+--------------------+-----------+---------------+-------------+-----------------+-----------+-----------------+------------------+--------------+------------+
|             trip_id|   route_id|route_long_name|stop_sequence|        stop_name|    stop_id|         stop_lon|          stop_lat|parent_station|direction_id|
+--------------------+-----------+---------------+-------------+-----------------+-----------+-----------------+------------------+--------------+------------+
|IDFM:RATP:174670-...|IDFM:C01387|             7B|            0|      Louis Blanc| IDFM:24681|2.364424862493678| 48.88120621182599|    IDFM:71407|           0|
|IDFM:RATP:174670-...|IDFM:C01387|             7B|            1|           Jaurès| IDFM:24680|2.370451734004561|48.882344230208226|    IDFM:71940|           0|
|IDFM:RATP:174670-...|IDFM:C01387|             7B|            2|          Bolivar| IDFM:24685|2.374124871187542| 48.88078966297506|    IDFM:71920|           0|
|IDFM:RATP:174670-...|IDFM:C01387|      

#### Creation of the graph

In [8]:
subway_stops_df = sql_context.sql("""
    SELECT DISTINCT 
        stop_id, 
        parent_station,
        stop_name
    FROM subway_trips
    ORDER BY stop_name
""")
subway_stops_df.createOrReplaceTempView("subway_stops")

mapping_df = sql_context.sql("""
    SELECT 
        stop_id, 
        COALESCE(parent_station, stop_id) as unique_stop_id,
        stop_name
    FROM subway_stops
""")
mapping_df.createOrReplaceTempView("stop_mapping")

vertices_df = sql_context.sql("""
    SELECT DISTINCT 
        m.unique_stop_id AS id, 
        m.stop_name
    FROM subway_trips t
    JOIN stop_mapping m ON t.stop_id = m.stop_id
    ORDER BY stop_name
""")

edges_unclean_df = sql_context.sql("""
    SELECT DISTINCT
        stop_id AS src,
        next_stop_id AS dst,
        route_id,
        direction_id
    FROM (
        SELECT 
            stop_id,
            LEAD(stop_id) OVER (PARTITION BY trip_id ORDER BY stop_sequence) AS next_stop_id,
            route_id,
            direction_id
        FROM subway_trips
    )
    WHERE next_stop_id IS NOT NULL
""")

edges_unclean_df.createOrReplaceTempView("edges_unclean")

edges_df = sql_context.sql("""
    SELECT
        sm.unique_stop_id as src,
        sm.stop_name as src_stop_name,
        dm.unique_stop_id as dst,
        dm.stop_name as dst_stop_name,
        e.route_id,
        e.direction_id
    FROM edges_unclean e
    JOIN stop_mapping sm ON e.src = sm.stop_id          
    JOIN stop_mapping dm ON e.dst = dm.stop_id 
    ORDER BY e.route_id         
""")

In [9]:
vertices_df.show(50)

+-----------+--------------------+
|         id|           stop_name|
+-----------+--------------------+
| IDFM:71432|            Abbesses|
| IDFM:72491|        Aimé Césaire|
| IDFM:71728|     Alexandre Dumas|
| IDFM:71290|      Alma - Marceau|
| IDFM:71030|              Alésia|
| IDFM:71519|      Anatole France|
| IDFM:71419|              Anvers|
| IDFM:71348|           Argentine|
| IDFM:71293|     Arts et Métiers|
| IDFM:72286|Asnières - Gennev...|
| IDFM:73642| Assemblée Nationale|
| IDFM:72460|Aubervilliers-Pan...|
| IDFM:71158|   Avenue Émile Zola|
| IDFM:71697|               Avron|
| IDFM:63284|     Aéroport d'Orly|
|IDFM:483315|Bagneux - Lucie A...|
| IDFM:71073|              Balard|
| IDFM:70441|             Barbara|
| IDFM:71426|Barbès - Rochecho...|
| IDFM:72326|Basilique de Sain...|
|IDFM:463564|            Bastille|
| IDFM:73650|             Bel-Air|
| IDFM:71853|          Belleville|
| IDFM:71607|               Bercy|
| IDFM:71572|Bibliothèque Fran...|
| IDFM:71034|       

In [10]:
edges_df.show(50)

+-----------+--------------------+-----------+--------------------+-----------+------------+
|        src|       src_stop_name|        dst|       dst_stop_name|   route_id|direction_id|
+-----------+--------------------+-----------+--------------------+-----------+------------+
| IDFM:71517|La Défense (Grand...| IDFM:71485|Esplanade de la D...|IDFM:C01371|           0|
| IDFM:71647|         Saint-Mandé| IDFM:71650|             Bérault|IDFM:C01371|           0|
| IDFM:71442|     Pont de Neuilly| IDFM:71485|Esplanade de la D...|IDFM:C01371|           1|
| IDFM:71485|Esplanade de la D...| IDFM:71442|     Pont de Neuilly|IDFM:C01371|           0|
| IDFM:71347|Charles de Gaulle...| IDFM:71328|            George V|IDFM:C01371|           0|
|IDFM:415852|      Hôtel de Ville| IDFM:71264|            Châtelet|IDFM:C01371|           1|
| IDFM:71328|            George V| IDFM:71347|Charles de Gaulle...|IDFM:C01371|           1|
| IDFM:71328|            George V| IDFM:71318|Franklin D. Roose...|IDF

In [11]:
from graphframes import GraphFrame

# Création du graphe
g = GraphFrame(vertices_df, edges_df)

g.cache() #A commenter si le cache a déjà été fait
g.vertices.cache()
g.edges.cache()

26/03/22 14:30:25 WARN CacheManager: Asked to cache already cached data.
26/03/22 14:30:25 WARN CacheManager: Asked to cache already cached data.


DataFrame[src: string, src_stop_name: string, dst: string, dst_stop_name: string, route_id: string, direction_id: int]

In [12]:
# Vérification
print(f"Nombre de stations : {g.vertices.count()}")
print(f"Nombre de connexions : {g.edges.count()}")

Nombre de stations : 321


Nombre de connexions : 773


##### Computations on the graph

In [13]:
g.vertices.show()

+-----------+--------------------+
|         id|           stop_name|
+-----------+--------------------+
| IDFM:71432|            Abbesses|
| IDFM:72491|        Aimé Césaire|
| IDFM:71728|     Alexandre Dumas|
| IDFM:71290|      Alma - Marceau|
| IDFM:71030|              Alésia|
| IDFM:71519|      Anatole France|
| IDFM:71419|              Anvers|
| IDFM:71348|           Argentine|
| IDFM:71293|     Arts et Métiers|
| IDFM:72286|Asnières - Gennev...|
| IDFM:73642| Assemblée Nationale|
| IDFM:72460|Aubervilliers-Pan...|
| IDFM:71158|   Avenue Émile Zola|
| IDFM:71697|               Avron|
| IDFM:63284|     Aéroport d'Orly|
|IDFM:483315|Bagneux - Lucie A...|
| IDFM:71073|              Balard|
| IDFM:70441|             Barbara|
| IDFM:71426|Barbès - Rochecho...|
| IDFM:72326|Basilique de Sain...|
+-----------+--------------------+
only showing top 20 rows



In [14]:
start_id = g.vertices.filter("stop_name = 'Abbesses'").select("id").first()[0]
end_id = g.vertices.filter("stop_name = 'Pigalle'").select("id").first()[0]

results = g.bfs(
    fromExpr=f"id = '{start_id}'",
    toExpr=f"id = '{end_id}'"
)
results.show(truncate=False)

+----------------------+-----------------------------------------------------------+---------------------+
|from                  |e0                                                         |to                   |
+----------------------+-----------------------------------------------------------+---------------------+
|{IDFM:71432, Abbesses}|{IDFM:71432, Abbesses, IDFM:71409, Pigalle, IDFM:C01382, 0}|{IDFM:71409, Pigalle}|
+----------------------+-----------------------------------------------------------+---------------------+



In [15]:
results = g.pageRank(
    resetProbability=0.15,  # probabilité de "random jump"
    maxIter=10              # nombre d’itérations
)

results.vertices.select("id", "stop_name", "pagerank") \
    .orderBy("pagerank", ascending=False) \
    .show(truncate=False)

+-----------+---------------------------+------------------+
|id         |stop_name                  |pagerank          |
+-----------+---------------------------+------------------+
|IDFM:71311 |République                 |2.8715732128914624|
|IDFM:71139 |Montparnasse Bienvenue     |2.713137196061611 |
|IDFM:71264 |Châtelet                   |2.6529121593170353|
|IDFM:71370 |Saint-Lazare               |2.319633209920964 |
|IDFM:71673 |Nation                     |2.264372389005422 |
|IDFM:71199 |La Motte-Picquet - Grenelle|2.2462972023402856|
|IDFM:70645 |Maison Blanche             |1.9965348109330396|
|IDFM:71347 |Charles de Gaulle - Etoile |1.993215289556499 |
|IDFM:463564|Bastille                   |1.9918326558082033|
|IDFM:71940 |Jaurès                     |1.9836999065085905|
|IDFM:71033 |Place d'Italie             |1.884698300490872 |
|IDFM:71961 |Stalingrad                 |1.875899130962189 |
|IDFM:71359 |Gare de l'Est              |1.7890181290350802|
|IDFM:71298 |Concorde   

In [19]:
degrees = g.degrees
vertices = g.vertices

result = degrees.join(vertices, on="id") \
                .select("id", "stop_name", "degree") \
                .orderBy("degree", ascending=False)

result.show(truncate=False)

+-----------+---------------------------+------+
|id         |stop_name                  |degree|
+-----------+---------------------------+------+
|IDFM:71311 |République                 |20    |
|IDFM:71264 |Châtelet                   |18    |
|IDFM:71370 |Saint-Lazare               |16    |
|IDFM:71139 |Montparnasse Bienvenue     |16    |
|IDFM:463564|Bastille                   |12    |
|IDFM:71673 |Nation                     |12    |
|IDFM:71298 |Concorde                   |12    |
|IDFM:71940 |Jaurès                     |12    |
|IDFM:71961 |Stalingrad                 |12    |
|IDFM:71337 |Opéra                      |12    |
|IDFM:73633 |Strasbourg - Saint-Denis   |12    |
|IDFM:71359 |Gare de l'Est              |12    |
|IDFM:71199 |La Motte-Picquet - Grenelle|12    |
|IDFM:71324 |Madeleine                  |12    |
|IDFM:71347 |Charles de Gaulle - Etoile |10    |
|IDFM:71033 |Place d'Italie             |10    |
|IDFM:70645 |Maison Blanche             |10    |
|IDFM:71435 |Place d

##### Télécharger les dépendances pour l'affichage

In [16]:
%pip instal pyvis

ERROR: unknown command "instal" - maybe you meant "install"
Note: you may need to restart the kernel to use updated packages.


In [17]:
from pyvis.network import Network
import IPython

# notebook=True est essentiel pour l'affichage direct dans Jupyter/Colab
# CDN_resources='in_line' évite les problèmes de chargement de scripts JS externes
net = Network(notebook=True, cdn_resources='in_line', height="700px", width="100%", bgcolor="#2d2d2d", font_color="white")

edges_pd = g.edges.filter("route_id = 'IDFM:C01371'").toPandas()
vertices_pd = g.vertices.toPandas()

# Ajouter les arêtes (Pyvis crée les sommets automatiquement)
for _, row in edges_pd.iterrows():
    net.add_node(row['src'], label=row['src']) # Vous pouvez remplacer par le nom de la station
    net.add_node(row['dst'], label=row['dst'])
    net.add_edge(row['src'], row['dst'])

# 1. On sauvegarde physiquement le fichier
net.save_graph("mon_graphe.html")

ModuleNotFoundError: No module named 'pyvis'